# Blog 4 — Incremental Data Processing
## Reliable Incremental ETL with Databricks, PySpark, Spark SQL and Delta Lake

This notebook builds a complete batch incremental pipeline for an e-commerce dataset.

We simulate:
- Day 1: 100,000 historical orders
- Day 2: 5,000 new orders
- Day 2: 1,000 updated orders
- Later: 1 late-arriving order

We cover and validate:
1. Full load
2. Watermark and control-table state
3. Incremental detection
4. Append vs MERGE
5. Source quality and deduplication
6. Watermark boundary conditions
7. Late-arriving data and a real lookback recovery
8. Schema drift
9. Failure after MERGE and retry
10. Idempotent recovery
11. A like-for-like Delta workload comparison
12. End-to-end reconciliation

The notebook is designed for Databricks Serverless and avoids Python RDD APIs and unavailable Serverless Spark configuration settings.

# 1. Architecture

```text
Source
  ↓
Bronze Delta
  ↓
Read watermark
  ↓
Incremental / lookback read
  ↓
Quality checks
  ↓
Deduplicate
  ↓
Delta MERGE
  ↓
Silver Delta
  ↓
Validate
  ↓
Update watermark
```

The key idea is:

> Process only new or changed data, apply it safely, and advance state only after successful validation.

# 2. Imports and configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timedelta
import time

NUM_INITIAL_ORDERS = 100_000
NUM_CUSTOMERS = 10_000
NEW_ORDERS = 5_000
UPDATED_ORDERS = 1_000

DAY1_TS = "2026-08-18 10:00:00"
DAY2_TS = "2026-08-19 10:00:00"

PIPELINE_NAME = "blog4_orders_incremental"
LOOKBACK_HOURS = 2

BRONZE = "blog4_bronze_orders"
SILVER = "blog4_silver_orders"
CONTROL = "blog4_etl_control"
BAD_APPEND = "blog4_bad_append"
IDEMPOTENT = "blog4_idempotent"
BENCH_FULL = "blog4_bench_full"
BENCH_INC = "blog4_bench_incremental"

print("Spark:", spark.version)

Spark: 4.1.0


# 3. Clean previous tutorial state

In [0]:
for t in [BRONZE, SILVER, CONTROL, BAD_APPEND, IDEMPOTENT, BENCH_FULL, BENCH_INC]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
print("Clean.")

Clean.


# 4. Day 1 — initial full load

In [0]:
orders_day1 = (
    spark.range(1, NUM_INITIAL_ORDERS + 1)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", ((F.col("order_id") * 17) % NUM_CUSTOMERS) + 1)
    .withColumn("amount", F.round(F.lit(100.0) + (F.col("order_id") % 500) * 10.0, 2))
    .withColumn("status", F.lit("COMPLETED"))
    .withColumn("updated_at", F.to_timestamp(F.lit(DAY1_TS)))
)

assert orders_day1.count() == NUM_INITIAL_ORDERS
display(orders_day1.limit(10))
print("PASS — 100,000 Day 1 records")

order_id,customer_id,amount,status,updated_at
1,18,110.0,COMPLETED,2026-08-18T10:00:00.000Z
2,35,120.0,COMPLETED,2026-08-18T10:00:00.000Z
3,52,130.0,COMPLETED,2026-08-18T10:00:00.000Z
4,69,140.0,COMPLETED,2026-08-18T10:00:00.000Z
5,86,150.0,COMPLETED,2026-08-18T10:00:00.000Z
6,103,160.0,COMPLETED,2026-08-18T10:00:00.000Z
7,120,170.0,COMPLETED,2026-08-18T10:00:00.000Z
8,137,180.0,COMPLETED,2026-08-18T10:00:00.000Z
9,154,190.0,COMPLETED,2026-08-18T10:00:00.000Z
10,171,200.0,COMPLETED,2026-08-18T10:00:00.000Z


PASS — 100,000 Day 1 records


# 5. Bronze and initial Silver

In [0]:
(orders_day1.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(BRONZE))
(spark.table(BRONZE).write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(SILVER))

assert spark.table(BRONZE).count() == 100_000
assert spark.table(SILVER).count() == 100_000
print("PASS — Bronze and Silver initialized")

PASS — Bronze and Silver initialized


# 6. Day 2 — new and updated records

In [0]:
new_orders = (
    spark.range(NUM_INITIAL_ORDERS + 1, NUM_INITIAL_ORDERS + NEW_ORDERS + 1)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", ((F.col("order_id") * 17) % NUM_CUSTOMERS) + 1)
    .withColumn("amount", F.round(F.lit(100.0) + (F.col("order_id") % 500) * 10.0, 2))
    .withColumn("status", F.lit("COMPLETED"))
    .withColumn("updated_at", F.to_timestamp(F.lit(DAY2_TS)))
)

updated_orders = (
    spark.range(1, UPDATED_ORDERS + 1)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", ((F.col("order_id") * 17) % NUM_CUSTOMERS) + 1)
    .withColumn("amount", F.round(F.lit(999.0) + (F.col("order_id") % 100) * 5.0, 2))
    .withColumn("status", F.lit("UPDATED"))
    .withColumn("updated_at", F.to_timestamp(F.lit(DAY2_TS)))
)

incremental_batch = new_orders.unionByName(updated_orders)

assert new_orders.count() == 5_000
assert updated_orders.count() == 1_000
assert incremental_batch.count() == 6_000

print("PASS — Day 2 contains 5,000 new + 1,000 updated")

PASS — Day 2 contains 5,000 new + 1,000 updated


# 7. Control table and watermark

In [0]:
control_df = spark.createDataFrame(
    [(PIPELINE_NAME, datetime.strptime(DAY1_TS, "%Y-%m-%d %H:%M:%S"), "SUCCESS")],
    "pipeline_name STRING, last_processed_timestamp TIMESTAMP, status STRING"
)

(control_df.write.format("delta").mode("overwrite").saveAsTable(CONTROL))

watermark = (
    spark.table(CONTROL)
    .filter(F.col("pipeline_name") == PIPELINE_NAME)
    .select("last_processed_timestamp")
    .first()["last_processed_timestamp"]
)

print("Watermark:", watermark)

Watermark: 2026-08-18 10:00:00


# 8. Incremental detection + boundary test

In [0]:
incremental = incremental_batch.filter(F.col("updated_at") > F.lit(watermark))

assert incremental.count() == 6_000

boundary = spark.createDataFrame(
    [(999999, 1, 500.0, "COMPLETED", watermark)],
    "order_id LONG, customer_id LONG, amount DOUBLE, status STRING, updated_at TIMESTAMP"
)

assert boundary.filter(F.col("updated_at") > F.lit(watermark)).count() == 0

print("PASS — 6,000 incremental records; exact watermark boundary excluded")

PASS — 6,000 incremental records; exact watermark boundary excluded


# 9. Why append is unsafe

In [0]:
(spark.table(BRONZE).write.format("delta").mode("overwrite").saveAsTable(BAD_APPEND))
incremental.write.format("delta").mode("append").saveAsTable(BAD_APPEND)

bad_count = spark.table(BAD_APPEND).count()
bad_dupes = (
    spark.table(BAD_APPEND).groupBy("order_id").count()
    .filter(F.col("count") > 1).count()
)

assert bad_count == 106_000
assert bad_dupes == 1_000

print("Append result:", bad_count, "rows")
print("Duplicate business keys:", bad_dupes)
print("PASS — append correctly demonstrated as unsafe")

Append result: 106000 rows
Duplicate business keys: 1000
PASS — append correctly demonstrated as unsafe


# 10. Source quality gates and negative duplicate test

In [0]:
null_keys = incremental.filter(F.col("order_id").isNull()).count()
null_ts = incremental.filter(F.col("updated_at").isNull()).count()
source_dupes = incremental.groupBy("order_id").count().filter(F.col("count") > 1).count()

assert null_keys == 0
assert null_ts == 0
assert source_dupes == 0

bad_source = spark.createDataFrame(
    [
        (1, 101, 500.0, "A", datetime.strptime(DAY2_TS,"%Y-%m-%d %H:%M:%S")),
        (1, 101, 700.0, "B", datetime.strptime(DAY2_TS,"%Y-%m-%d %H:%M:%S"))
    ],
    "order_id LONG, customer_id LONG, amount DOUBLE, status STRING, updated_at TIMESTAMP"
)

bad_source_dupes = bad_source.groupBy("order_id").count().filter(F.col("count") > 1).count()
assert bad_source_dupes == 1

print("PASS — null/duplicate quality gates")
print("PASS — deliberate duplicate-source negative test")

PASS — null/duplicate quality gates
PASS — deliberate duplicate-source negative test


# 11. Deduplicate before MERGE

In [0]:
w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())

deduped = (
    incremental
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

assert deduped.count() == 6_000
print("PASS — source is one row per business key")

PASS — source is one row per business key


# 12. Delta MERGE — insert new, update existing

In [0]:
deduped.createOrReplaceTempView("blog4_source")

spark.sql(f"""
MERGE INTO {SILVER} AS target
USING blog4_source AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET
    target.customer_id = source.customer_id,
    target.amount = source.amount,
    target.status = source.status,
    target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT
    (order_id, customer_id, amount, status, updated_at)
VALUES
    (source.order_id, source.customer_id, source.amount, source.status, source.updated_at)
""")

assert spark.table(SILVER).count() == 105_000
print("PASS — MERGE produced 105,000 rows")

PASS — MERGE produced 105,000 rows


# 13. Validate inserts, updates and uniqueness

In [0]:
insert_count = spark.table(SILVER).filter(
    F.col("order_id").between(100001, 105000)
).count()

update_count = spark.table(SILVER).filter(
    (F.col("order_id").between(1,1000)) &
    (F.col("status") == "UPDATED") &
    (F.col("updated_at") == F.to_timestamp(F.lit(DAY2_TS)))
).count()

duplicate_target = spark.table(SILVER).groupBy("order_id").count().filter(F.col("count") > 1).count()

assert insert_count == 5_000
assert update_count == 1_000
assert duplicate_target == 0

print("Inserted:", insert_count)
print("Updated:", update_count)
print("Duplicate target keys:", duplicate_target)
print("PASS — Silver validation")

Inserted: 5000
Updated: 1000
Duplicate target keys: 0
PASS — Silver validation


# 14. Late-arriving data — reproduce the failure of a strict watermark

Assume the current processing boundary is 10:00.

A record arrives later but has event/update time 09:30.

Strict:

```text
updated_at > 10:00
```

misses it.

We then use a 2-hour lookback:

```text
10:00 - 2 hours = 08:00
```

so 09:30 is recovered.

In [0]:
demo_watermark = datetime.strptime(DAY2_TS, "%Y-%m-%d %H:%M:%S")

late_order = spark.createDataFrame(
    [
        (105001, 5001, 750.0, "COMPLETED", "2026-08-19 09:30:00", "2026-08-20 10:00:00")
    ],
    "order_id LONG, customer_id LONG, amount DOUBLE, status STRING, updated_at STRING, arrival_time STRING"
).withColumn("updated_at", F.to_timestamp("updated_at")).withColumn("arrival_time", F.to_timestamp("arrival_time"))

strict_late = late_order.filter(F.col("updated_at") > F.lit(demo_watermark))
assert strict_late.count() == 0

lookback_start = demo_watermark - timedelta(hours=LOOKBACK_HOURS)
late_recovered = late_order.filter(F.col("updated_at") >= F.lit(lookback_start))

assert late_recovered.count() == 1

print("Strict watermark captures:", strict_late.count())
print("Lookback captures:", late_recovered.count())
print("Lookback start:", lookback_start)

Strict watermark captures: 0
Lookback captures: 1
Lookback start: 2026-08-19 08:00:00


# 15. Actually MERGE the late-arriving record

In [0]:
late_recovered.createOrReplaceTempView("blog4_late_source")

spark.sql(f"""
MERGE INTO {SILVER} AS target
USING blog4_late_source AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET
    target.customer_id = source.customer_id,
    target.amount = source.amount,
    target.status = source.status,
    target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT
    (order_id, customer_id, amount, status, updated_at)
VALUES
    (source.order_id, source.customer_id, source.amount, source.status, source.updated_at)
""")

assert spark.table(SILVER).count() == 105_001

late_exists = spark.table(SILVER).filter(F.col("order_id") == 105001).count()
assert late_exists == 1

print("PASS — late record merged")
print("Silver:", spark.table(SILVER).count())
print("Late record count:", late_exists)

PASS — late record merged
Silver: 105001
Late record count: 1


# 16. Schema-drift test

## What counts as a real schema-drift test?

Simply checking `df.columns` is not enough.

We will:

1. Create a real Delta target with the original Silver schema.
2. Create a source with an extra `source_region` column.
3. Attempt an append **without** schema evolution and capture the expected failure.
4. Then write the same data with `mergeSchema=true` to a separate demonstration table.
5. Verify that the new column actually exists.

We keep this demonstration table separate from the main Silver table so the core pipeline schema remains explicit and controlled.

In [0]:

SCHEMA_DRIFT_TABLE = "blog4_schema_drift_demo"
spark.sql(f"DROP TABLE IF EXISTS {SCHEMA_DRIFT_TABLE}")

schema_drift = deduped.withColumn("source_region", F.lit("SOUTH"))

# Create the target with the original schema.
(
    deduped.limit(10)
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(SCHEMA_DRIFT_TABLE)
)

assert "source_region" not in spark.table(SCHEMA_DRIFT_TABLE).columns
assert "source_region" in schema_drift.columns

print("Target columns before drift:", spark.table(SCHEMA_DRIFT_TABLE).columns)
print("Source columns:", schema_drift.columns)


Target columns before drift: ['order_id', 'customer_id', 'amount', 'status', 'updated_at']
Source columns: ['order_id', 'customer_id', 'amount', 'status', 'updated_at', 'source_region']


In [0]:

# Attempt the schema-drift write WITHOUT mergeSchema.
# We expect Delta to reject the extra source column.

schema_drift_failed = False

try:
    (
        schema_drift.limit(10)
        .write.format("delta")
        .mode("append")
        .saveAsTable(SCHEMA_DRIFT_TABLE)
    )
except Exception as e:
    schema_drift_failed = True
    print("Expected schema-drift failure:")
    print(type(e).__name__)
    print(str(e)[:1000])

assert schema_drift_failed
print("PASS — real write failed without schema evolution")


Expected schema-drift failure:
AnalysisException
[DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 35d30fe0-8fe9-4d4c-9e28-e4b82b710f1f).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)


Data schema:
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- source_region: 

In [0]:

# Now explicitly enable schema evolution on the demonstration table.

(
    schema_drift.limit(10)
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SCHEMA_DRIFT_TABLE)
)

drift_columns_after = spark.table(SCHEMA_DRIFT_TABLE).columns

assert "source_region" in drift_columns_after

print("Columns after mergeSchema:", drift_columns_after)
print("PASS — schema evolution succeeded only when explicitly enabled")


Columns after mergeSchema: ['order_id', 'customer_id', 'amount', 'status', 'updated_at', 'source_region']
PASS — schema evolution succeeded only when explicitly enabled


### Lesson

`mergeSchema=true` is not a generic fix for every schema problem.

Schema evolution should be an intentional contract decision. The main Silver pipeline in this notebook continues to use explicit MERGE mappings so an unexpected source column does not silently become part of the production contract.

# 17. Real failure injection after MERGE, before state update

# 17. Failure and recovery

There are two different failure cases worth separating:

### A. Failure inside the Delta MERGE statement

We can reproduce this safely by supplying an invalid/ambiguous source and showing that the Delta transaction does not leave a partially applied target.

### B. Failure after the MERGE commits but before pipeline state is advanced

We can reproduce this exactly with a deliberate exception. This is the important retry/idempotency boundary.

We should **not** claim that a Python exception simulates a physical executor crash in the middle of a Delta commit. Delta's transaction protocol is what protects the target from partial commits.

In [0]:

# A. Failure in the MERGE operation itself.
# Start with a clean 100,000-row target.

MERGE_FAILURE_TABLE = "blog4_merge_failure_demo"
spark.sql(f"DROP TABLE IF EXISTS {MERGE_FAILURE_TABLE}")

(
    spark.table(BRONZE)
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(MERGE_FAILURE_TABLE)
)

bad_merge_source = spark.createDataFrame(
    [
        (1, 101, 500.0, "BAD_A", datetime.strptime(DAY2_TS,"%Y-%m-%d %H:%M:%S")),
        (1, 101, 700.0, "BAD_B", datetime.strptime(DAY2_TS,"%Y-%m-%d %H:%M:%S"))
    ],
    "order_id LONG, customer_id LONG, amount DOUBLE, status STRING, updated_at TIMESTAMP"
)

bad_merge_source.createOrReplaceTempView("blog4_bad_merge_source")

merge_failed = False

try:
    spark.sql(f"""
    MERGE INTO {MERGE_FAILURE_TABLE} AS target
    USING blog4_bad_merge_source AS source
    ON target.order_id = source.order_id
    WHEN MATCHED THEN UPDATE SET
        target.customer_id = source.customer_id,
        target.amount = source.amount,
        target.status = source.status,
        target.updated_at = source.updated_at
    WHEN NOT MATCHED THEN INSERT
        (order_id, customer_id, amount, status, updated_at)
    VALUES
        (source.order_id, source.customer_id, source.amount, source.status, source.updated_at)
    """)
except Exception as e:
    merge_failed = True
    print("Expected MERGE failure:")
    print(type(e).__name__)
    print(str(e)[:1200])

assert merge_failed

# Atomicity check: the failed MERGE did not partially apply the source.
post_failure_count = spark.table(MERGE_FAILURE_TABLE).count()
assert post_failure_count == 100_000

print("PASS — failed MERGE left target unchanged")


Expected MERGE failure:
UnsupportedOperationException
[DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE] Cannot perform Merge as multiple source rows matched and attempted to modify the same
target row in the Delta table in possibly conflicting ways. By SQL semantics of Merge,
when multiple source rows match on the same target row, the result may be ambiguous
as it is unclear which source row should be used to update or delete the matching
target row. You can preprocess the source table to eliminate the possibility of
multiple matches. Please refer to
https://docs.databricks.com/delta/merge.html#merge-error


JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaUnsupportedOperationException
	at com.databricks.sql.transaction.tahoe.DeltaErrorsEdge.multipleSourceRowMatchingTargetRowInMergeException(DeltaErrorsEdge.scala:1396)
	at com.databricks.sql.transaction.tahoe.DeltaErrorsEdge.multipleSourceRowMatchingTargetRowInMergeException$(DeltaErrorsEdge.scala:1391)
	at com.databric

### What this proves

The duplicate source is rejected by the MERGE operation, and the target remains at 100,000 rows.

This is a better demonstration of failure during the write path than merely raising an exception after a successful MERGE.

# 18. Partial-failure boundary — MERGE succeeds, state update fails

Now test the operational failure that matters for incremental pipelines:

```text
MERGE commits
   ↓
state update fails
   ↓
watermark remains old
   ↓
same batch is retried
   ↓
MERGE must be safe
```

In [0]:

(
    spark.table(BRONZE)
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(IDEMPOTENT)
)

failure_batch = incremental
failure_batch.createOrReplaceTempView("blog4_failure_source")

spark.sql(f"""
MERGE INTO {IDEMPOTENT} AS target
USING blog4_failure_source AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET
    target.customer_id = source.customer_id,
    target.amount = source.amount,
    target.status = source.status,
    target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT
    (order_id, customer_id, amount, status, updated_at)
VALUES
    (source.order_id, source.customer_id, source.amount, source.status, source.updated_at)
""")

assert spark.table(IDEMPOTENT).count() == 105_000

# Simulate the pipeline's state update failing.
state_update_failed = False

try:
    raise RuntimeError("SIMULATED FAILURE: control-table update failed after committed MERGE")
except RuntimeError as e:
    state_update_failed = True
    print("Expected state-update failure:")
    print(e)

assert state_update_failed

# The watermark would still point to Day 1.
assert watermark == datetime.strptime(DAY1_TS,"%Y-%m-%d %H:%M:%S")

print("PASS — committed target + old watermark represents a recoverable retry boundary")


Expected state-update failure:
SIMULATED FAILURE: control-table update failed after committed MERGE
PASS — committed target + old watermark represents a recoverable retry boundary


# 19. Retry the exact same batch — idempotent recovery

In [0]:

# Retry the exact same 6,000 records.

spark.sql(f"""
MERGE INTO {IDEMPOTENT} AS target
USING blog4_failure_source AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET
    target.customer_id = source.customer_id,
    target.amount = source.amount,
    target.status = source.status,
    target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT
    (order_id, customer_id, amount, status, updated_at)
VALUES
    (source.order_id, source.customer_id, source.amount, source.status, source.updated_at)
""")

retry_count = spark.table(IDEMPOTENT).count()
retry_dupes = (
    spark.table(IDEMPOTENT)
    .groupBy("order_id").count()
    .filter(F.col("count") > 1).count()
)

assert retry_count == 105_000
assert retry_dupes == 0

print("After retry:", retry_count)
print("Duplicate keys:", retry_dupes)
print("PASS — exact-batch retry is idempotent")


After retry: 105000
Duplicate keys: 0
PASS — exact-batch retry is idempotent


### Important limitation

This does **not** simulate a machine crash halfway through a Delta transaction.

Instead, it tests the two boundaries we can safely reproduce in a notebook:

- a MERGE statement fails and leaves the target unchanged
- a MERGE commits but the state update fails, so the same batch is retried

The second case is the key pipeline-level idempotency scenario.

# 20. Why this is idempotent

The same 6,000 records were applied twice.

The final target stayed at 105,000.

```text
First attempt → 105,000
Failure       → state not advanced
Retry         → same 6,000 input
Retry result  → 105,000
```

This is why incremental processing and idempotency are closely connected.

MERGE helps, but pipeline-level idempotency also requires stable keys, deterministic transformations, safe side effects and controlled state.

# 21. Safe watermark update

In [0]:
new_watermark = (
    failure_batch
    .agg(F.max("updated_at").alias("max_updated_at"))
    .first()["max_updated_at"]
)

print("Only after successful MERGE + validation, advance to:", new_watermark)

Only after successful MERGE + validation, advance to: 2026-08-19 10:00:00


# 21. Fairer workload comparison

# 22. Full-load vs incremental workload — compare the actual Day 2 work

A proper conceptual comparison is:

### Full Day 2 processing

The pipeline reprocesses the complete current source snapshot.

### Incremental Day 2 processing

The pipeline processes only:

```text
5,000 new + 1,000 updates = 6,000 rows
```

To make the comparison meaningful, we construct a current Day 2 source snapshot of 105,000 rows:

```text
Day 1 rows excluding the 1,000 updated keys
+ 1,000 updated rows
+ 5,000 new rows
= 105,000 current rows
```

Then we materialize both workloads as Delta tables.

We will use row volume as the primary teaching metric. We will not pretend that a tiny notebook timing is a reliable performance benchmark.

In [0]:

# Build the current Day 2 full-source snapshot.

day2_full_snapshot = (
    orders_day1
    .filter(~F.col("order_id").between(1, UPDATED_ORDERS))
    .unionByName(updated_orders)
    .unionByName(new_orders)
)

assert day2_full_snapshot.count() == 105_000
assert day2_full_snapshot.select("order_id").distinct().count() == 105_000

print("Day 2 full snapshot:", day2_full_snapshot.count())
print("Day 2 incremental batch:", incremental.count())


Day 2 full snapshot: 105000
Day 2 incremental batch: 6000


In [0]:

(
    day2_full_snapshot.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(BENCH_FULL)
)

(
    incremental.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(BENCH_INC)
)

full_rows = spark.table(BENCH_FULL).count()
inc_rows = spark.table(BENCH_INC).count()

assert full_rows == 105_000
assert inc_rows == 6_000

row_reduction_pct = (full_rows - inc_rows) / full_rows * 100

print("Full Day 2 workload:", full_rows)
print("Incremental Day 2 workload:", inc_rows)
print(f"Rows avoided by incremental processing: {row_reduction_pct:.2f}%")

assert round(row_reduction_pct, 2) == 94.29


Full Day 2 workload: 105000
Incremental Day 2 workload: 6000
Rows avoided by incremental processing: 94.29%


### Why we do not make a runtime claim

At this dataset size, Spark/Databricks fixed overhead can dominate execution time. Two tiny Delta scans can easily have similar timings even when one contains far fewer rows.

Therefore:

> **94.29% fewer rows is a workload-volume result, not a claim of 94.29% lower runtime.**

For production performance work, benchmark the actual end-to-end pipeline with representative table sizes, file layouts, cluster/Photon settings and repeated runs.

In [0]:

# Optional exploratory timing — deliberately labeled as non-benchmark evidence.

def timed_count(table_name, label):
    start = time.perf_counter()
    n = spark.table(table_name).count()
    elapsed = time.perf_counter() - start
    print(f"{label}: {n:,} rows | {elapsed:.3f}s")
    return elapsed

print("These timings are illustrative only.")
full_seconds = timed_count(BENCH_FULL, "Full Day 2 snapshot scan")
inc_seconds = timed_count(BENCH_INC, "Incremental batch scan")

print("Do not infer a linear runtime relationship from these numbers.")


These timings are illustrative only.
Full Day 2 snapshot scan: 105,000 rows | 0.290s
Incremental batch scan: 6,000 rows | 0.299s
Do not infer a linear runtime relationship from these numbers.


# 23. Physical plan inspection

In [0]:
print("=== FULL DELTA PLAN ===")
spark.table(BENCH_FULL).explain("formatted")

print("=== INCREMENTAL DELTA PLAN ===")
spark.table(BENCH_INC).explain("formatted")

=== FULL DELTA PLAN ===
== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet workspace.default.blog4_bench_full (1)


(1) PhotonScan parquet workspace.default.blog4_bench_full
Output [5]: [order_id#30488L, customer_id#30489L, amount#30490, status#30491, updated_at#30492]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-wozmd/uc/86c84868-9c99-493b-89e8-6d4f65f53aa5/7f44b365-14cf-4b2d-aa61-7e607e72a13d/__unitystorage/catalogs/6f781c3e-1995-4c2c-b009-edbc1f2fc8a4/tables/27133b2d-3d01-4174-9cb1-b0d9b7bf332e]
ReadSchema: struct<order_id:bigint,customer_id:bigint,amount:double,status:string,updated_at:timestamp>

(2) PhotonColumnarToRow
Input [5]: [order_id#30488L, customer_id#30489L, amount#30490, status#30491, updated_at#30492]

(3) PhotonResultStage
Input [5]: [order_id#30488L, customer_id#30489L, amount#30490, status#30491, updated_at#30492]


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table name

# 24. Final reconciliation

In [0]:
final_count = spark.table(SILVER).count()

original_count = spark.table(SILVER).filter(
    F.col("order_id").between(1,100000)
).count()

new_count = spark.table(SILVER).filter(
    F.col("order_id").between(100001,105000)
).count()

updated_count = spark.table(SILVER).filter(
    (F.col("order_id").between(1,1000)) &
    (F.col("status") == "UPDATED")
).count()

late_count = spark.table(SILVER).filter(
    F.col("order_id") == 105001
).count()

duplicate_count = spark.table(SILVER).groupBy("order_id").count().filter(
    F.col("count") > 1
).count()

assert final_count == 105_001
assert original_count == 100_000
assert new_count == 5_000
assert updated_count == 1_000
assert late_count == 1
assert duplicate_count == 0

print("Final Silver:", final_count)
print("Original retained:", original_count)
print("New inserted:", new_count)
print("Updated:", updated_count)
print("Late-arriving:", late_count)
print("Duplicate keys:", duplicate_count)
print("PASS — end-to-end reconciliation")

Final Silver: 105001
Original retained: 100000
New inserted: 5000
Updated: 1000
Late-arriving: 1
Duplicate keys: 0
PASS — end-to-end reconciliation


# 25. Final control-table state

In [0]:
main_watermark = (
    deduped.agg(F.max("updated_at").alias("max_updated_at")).first()["max_updated_at"]
)

spark.sql(f"""
UPDATE {CONTROL}
SET last_processed_timestamp = TIMESTAMP('{main_watermark}'),
    status = 'SUCCESS'
WHERE pipeline_name = '{PIPELINE_NAME}'
""")

state = spark.table(CONTROL).filter(F.col("pipeline_name") == PIPELINE_NAME).first()

assert state["status"] == "SUCCESS"
assert state["last_processed_timestamp"] == datetime.strptime(DAY2_TS,"%Y-%m-%d %H:%M:%S")

display(spark.table(CONTROL))
print("PASS — watermark advanced only after successful processing")

pipeline_name,last_processed_timestamp,status
blog4_orders_incremental,2026-08-19T10:00:00.000Z,SUCCESS


PASS — watermark advanced only after successful processing


# 26. Production lessons

### Incremental detection
Use a watermark, CDC, CDF, sequence number or another reliable change mechanism.

### State
Persist the last successful processing point in a control table.

### Upsert
Use a stable business key and Delta MERGE for update/insert behavior.

### Data quality
Validate null keys, required fields and duplicate source keys before MERGE.

### Late data
Use a justified lookback window when late arrival is possible.

### Retry safety
Design the pipeline so the same batch can be processed again safely.

### Validation
Reconcile the actual business state instead of relying only on successful job execution.

### Performance
Measure real workloads. Fewer rows is useful, but row reduction is not the same thing as runtime reduction.

# 27. Interview-ready explanation

> I built an incremental Delta Lake ETL pipeline in Databricks using PySpark and Spark SQL. The pipeline stores a high-water mark in a control table, detects new and changed records, validates and deduplicates the source, and uses Delta MERGE to update existing orders and insert new ones. I explicitly tested the watermark boundary, recovered a late-arriving event using a two-hour lookback and merged it into Silver, then injected a failure after MERGE but before the state update and showed that retrying the same batch remained idempotent. Finally, I reconciled inserts, updates, late data and duplicate keys and separated row-volume reduction from runtime claims.

# 28. Final checklist

- [x] Full initial load
- [x] Bronze and Silver Delta tables
- [x] 5,000 new records
- [x] 1,000 updated records
- [x] Watermark
- [x] Exact watermark boundary test
- [x] Persistent control table
- [x] Append-vs-MERGE demonstration
- [x] Null-key and timestamp quality checks
- [x] Duplicate-source negative test
- [x] Source deduplication
- [x] Delta MERGE
- [x] Insert/update/uniqueness validation
- [x] Late-arriving data
- [x] Real lookback window
- [x] Late record actually merged
- [x] Actual schema-drift write failure test
- [x] Explicit schema evolution demonstration
- [x] MERGE failure + atomicity check
- [x] Post-MERGE state-update failure
- [x] Exact-batch retry
- [x] Idempotent recovery validation
- [x] Full Day 2 snapshot vs incremental workload comparison
- [x] 94.29% row-volume reduction
- [x] Honest small-data timing caveat
- [x] Physical-plan inspection
- [x] Final reconciliation
- [x] Final control-table validation

## Blog 4 is complete.

# 29. Optional cleanup

For screenshots and blog validation, keep the tables.

When finished:

```python
for t in [BRONZE, SILVER, CONTROL, BAD_APPEND, IDEMPOTENT, BENCH_FULL, BENCH_INC, SCHEMA_DRIFT_TABLE, MERGE_FAILURE_TABLE]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
```